# 08 — Guest Experience Business Insights
Combine sentiment, topics, hotels, segments and booking context into actionable reports.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
for d in (FIGURES_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SENTIMENT_LABELS = ["negative", "neutral", "positive"]

def load_any(stem: Path):
    for s in (".parquet", ".pkl"):
        p = stem.with_suffix(s)
        if p.exists() and p.stat().st_size > 0:
            return pd.read_parquet(p) if s == ".parquet" else pd.read_pickle(p)
    return None

df = load_any(DATA_DIR / "cleaned_reviews")
assert df is not None, "Run 02_data_cleaning first"
print("Reviews:", len(df))


In [ ]:
# Hotel performance leaderboard
hotel_col = "ht_hotel_name" if "ht_hotel_name" in df.columns else "hotel_id"
loc_col = "ht_location" if "ht_location" in df.columns else None

group_cols = [hotel_col] + ([loc_col] if loc_col else [])
hotel_sum = (
    df.groupby(group_cols)
    .agg(
        n_reviews=("review_id", "count"),
        mean_rating=("rating_score", "mean"),
        pct_positive=("sentiment_label", lambda x: (x == "positive").mean() * 100),
        pct_negative=("sentiment_label", lambda x: (x == "negative").mean() * 100),
    )
    .round(2)
    .reset_index()
    .sort_values("mean_rating")
)
display(hotel_sum)


In [ ]:
# Hotel leaderboard chart
fig, ax = plt.subplots(figsize=(10, 8))
plot_df = hotel_sum.sort_values("mean_rating")
colors = ["#e74c3c" if r < 3.0 else "#f39c12" if r < 3.8 else "#27ae60" for r in plot_df["mean_rating"]]
ax.barh(plot_df[hotel_col], plot_df["mean_rating"], color=colors)
ax.set_xlabel("Average Rating")
ax.set_title("Hotel Rating Leaderboard")
ax.axvline(plot_df["mean_rating"].mean(), color="grey", linestyle="--", label="Overall mean")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "11_hotel_leaderboard.png", dpi=150)
plt.show()


In [ ]:
# Sentiment by guest segment
for col in ["gt_customer_type", "gt_loyalty_status", "bk_booking_channel", "bk_room_type"]:
    if col not in df.columns:
        continue
    ct = (pd.crosstab(df[col], df["sentiment_label"], normalize="index") * 100).round(1)
    ct["n"] = df.groupby(col).size()
    print(f"\n=== {col} ===")
    display(ct)


In [ ]:
# Trends over time
if "review_date" in df.columns:
    ts = df.set_index("review_date").sort_index()
    monthly = ts.resample("ME").agg(
        n_reviews=("review_id", "count"),
        mean_rating=("rating_score", "mean"),
        pct_positive=("sentiment_label", lambda x: (x == "positive").mean() * 100),
        pct_negative=("sentiment_label", lambda x: (x == "negative").mean() * 100),
    ).round(2)

    fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
    monthly["mean_rating"].plot(ax=axes[0], marker="o", color="#3498db")
    axes[0].set_ylabel("Mean Rating")
    axes[0].set_title("Rating & Sentiment Trends Over Time")
    monthly[["pct_positive", "pct_negative"]].plot(ax=axes[1], marker="o")
    axes[1].set_ylabel("% of Reviews")
    axes[1].legend(["Positive %", "Negative %"])
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "12_sentiment_trends.png", dpi=150)
    plt.show()
    display(monthly.tail(8))


In [ ]:
# Optional: merge topic sentiment if available
topic_sent = None
topics = load_any(MODELS_DIR / "document_topics")
if topics is not None:
    merged = df.merge(topics, on="review_id", how="inner")
    work = merged[merged["topic_id"] != -1]
    topic_sent = (
        work.groupby(["topic_id", "topic_label"])
        .agg(
            n_docs=("sentiment_label", "size"),
            pct_negative=("sentiment_label", lambda x: (x == "negative").mean() * 100),
            pct_positive=("sentiment_label", lambda x: (x == "positive").mean() * 100),
            mean_rating=("rating_score", "mean"),
        )
        .round(2)
        .reset_index()
        .sort_values("n_docs", ascending=False)
    )
    print("Top topics by volume:")
    display(topic_sent.head(12))
else:
    print("No topic mapping found – run 04_topic_modeling for topic insights.")


In [ ]:
# Write business insight report
report_path = REPORTS_DIR / "business_insight_report.md"
with open(report_path, "w") as f:
    f.write("# Guest Experience Business Insight Report\n")
    f.write("## Montra Hospitality Intel\n\n")
    f.write("### 1. Executive Summary\n\n")
    f.write(f"- Total reviews analysed: **{len(df):,}**\n")
    f.write(f"- Overall mean rating: **{df['rating_score'].mean():.2f}**\n")
    sent_pct = (df["sentiment_label"].value_counts(normalize=True) * 100).round(1)
    for s in SENTIMENT_LABELS:
        f.write(f"- {s.capitalize()}: **{sent_pct.get(s, 0)}%**\n")
    f.write("\n### 2. Hotel Performance Leaderboard\n\n")
    f.write(hotel_sum.to_markdown(index=False))
    f.write("\n\n### 3. Recommendations\n\n")
    f.write("1. Focus operational resources on hotels/topics with the highest negative share.\n")
    f.write("2. Investigate root causes behind low-rated properties.\n")
    f.write("3. Share positive-topic best practices across the portfolio.\n")
    f.write("4. Monitor monthly sentiment trends for early warning of emerging issues.\n")
    f.write("5. Tailor service recovery for high-value loyalty tiers.\n")

print(f"Report written → {report_path}")
print("\nDone – all insight figures and report are under outputs/")
